## StudyChat dialogue-act annotator + outcome analysis (sibling of check_sharechat / check_wildchat)

Real students in UMass **COMPSCI 383 (AI)**. They use an LLM assistant on **assignments** (logged,
LLM-allowed) then take **proctored exams** (no LLM). The grade file gives per-student aggregates:
`llm` = assisted-assignment average, `no_llm` = exam average — i.e. the **phase-1 structure in the
wild**: assisted work (R1) -> unassisted transfer (R2), with objective grades as the outcome.

Population = the **181 consented students** with released grades (f24=70, s25=111). No pedagogical
judge needed (it is all coursework). We annotate their assignment user-turns with OUR 5-act taxonomy
and test whether R1 reasoning predicts exam (no_llm) transfer, controlling for assignment (llm)
performance — the same OLS as phase-1. (StudyChat's own `llm_label` is kept as `their_label` for cross-check.)

Cells: (A) grades, (B) load+filter conversations, (1) taxonomy, (2) signature, (3) suite,
(4) annotate user turns (OUR taxonomy), (5) outcome OLS + coverage.

In [1]:
import pandas as pd, pathlib, dotenv
dotenv.load_dotenv(pathlib.Path("..") / "wildchat_analysis" / ".env")   # HF_TOKEN (gated dataset)
from huggingface_hub import hf_hub_download
REPO = "wmcnicho/StudyChat"
_g = lambda f: hf_hub_download(REPO, f, repo_type="dataset")
_f24 = pd.read_csv(_g("scores/f24_grades_released_normalized.csv")); _f24["semester"] = "f24"
_s25 = pd.read_csv(_g("scores/s25_grades_released_normalized.csv")); _s25["semester"] = "s25"
grades = pd.concat([_f24, _s25], ignore_index=True)
consented = set(grades["userId"])
print(f"consented students with grades: {len(grades)}  (f24 {len(_f24)}, s25 {len(_s25)})")
print("outcome cols -> llm = assisted-assignment avg (R1) ; no_llm = exam avg (R2 transfer)")
grades[["userId", "semester", "llm", "no_llm"]].head()

consented students with grades: 181  (f24 70, s25 111)
outcome cols -> llm = assisted-assignment avg (R1) ; no_llm = exam avg (R2 transfer)


,userId,semester,llm,no_llm
0,d1db45a0-f051-702b-f627-c33637d4528c,f24,0.946667,0.958333
1,c18b0590-1021-70b2-a04d-7da82bf20079,f24,0.908571,0.720000
2,613b1510-4051-7027-1ee2-8f2c72d3c64b,f24,0.944762,0.850000
3,315b9550-4081-7033-4f93-fe2be67cdfcc,f24,0.965714,0.750000
4,619bf560-4021-7015-9155-f148328587b8,f24,0.984762,1.000000


In [2]:
# StudyChat is per-INTERACTION: each row = one user prompt (+ assistant response) in a chat.
# Stream data.jsonl (the 'combined' config, ~626MB, downloaded+cached once), keep consented
# students, and treat each row's `prompt` as one user turn to annotate.
from datasets import load_dataset

_ds = load_dataset(REPO, split="train", streaming=True)
rows = []
for ex in _ds:
    if ex.get("userId") not in consented:
        continue
    p = ex.get("prompt")
    if not isinstance(p, str) or not p.strip():
        continue
    rows.append({"userId": ex["userId"], "chatId": ex.get("chatId"), "topic": ex.get("topic"),
                 "prompt": p, "semester": ex.get("semester"),
                 "their_label": (ex.get("llm_label") or {}).get("label")})
turns_df = pd.DataFrame(rows)
print(f"{len(turns_df)} user turns from {turns_df['userId'].nunique()} consented students")
print("median user turns/student:", int(turns_df.groupby('userId').size().median()))
print("turns by assignment:", dict(turns_df['topic'].value_counts()))

KeyboardInterrupt: 

In [ ]:
# (1) Config + taxonomy tables (copied from phase-2 dialogue_act_annotation.py)
import concurrent.futures, logging, pathlib
from collections import Counter
from typing import Literal
import dspy, dotenv

# OPENAI / TOGETHER keys live in the wildchat_analysis/.env; load them here.
dotenv.load_dotenv()
dotenv.load_dotenv(pathlib.Path("..") / "wildchat_analysis" / ".env")

_log = logging.getLogger("dialogue_act_annotation")

# Scheme membership (T = Tutor move, S = Student move)
SCHEME = {
    "Think Aloud": "S", "Conversational Acknowledgment": "S",
    "Knowledge Deficit Question": "S", "Misconception": "S",
    "Common Ground Question": "S", "Vague Answer": "S", "Partial Answer": "S",
    "Social Coordination Action": "S", "Metacomment": "S", "Read Aloud": "S",
    "Solution Request": "S",
    "Forced Choice": "T", "Repetition": "T", "Prompt": "T",
}

# In-context examples per move (from Table I; Solution Request added for human<->AI).
EXAMPLES = {
    "Forced Choice": '"Would that be random, uniformed, or clumped?"',
    "Prompt": '"So 200 times one is what?"',
    "Repetition": 'S: "Is it commensalism?" T: "Commensalism."',
    "Common Ground Question": '"Aren\'t they more lined up, like more in order?"',
    "Conversational Acknowledgment": '"Ok." "No sir." "Yes ma\'am."',
    "Knowledge Deficit Question": '"What do you mean by it doesn\'t have a skeleton?"',
    "Metacomment": '"I don\'t know." "Yes, I understand."',
    "Misconception": '"I always used to get diploid and haploid mixed up."',
    "Partial Answer": '"It has to do with the cells."',
    "Read Aloud": '"Question 7: Plot growth pattern."',
    "Social Coordination Action": '"No, I didn\'t hear about that."',
    "Think Aloud": '"500 equals 50 and 50 divided by 500 gives 10."',
    "Vague Answer": '"Because it helps to, umm, you know."',
    "Solution Request": '"What\'s the best move?" "What now?" "Just tell me what to play." "Any advice for the next move?"',
}

TAXONOMY = ""
for k in SCHEME:
    TAXONOMY += "DIALOGUE ACT: " + k + ", EXAMPLES: " + EXAMPLES[k] + "\n"

DialogueAct = Literal[
    'Think Aloud', 'Conversational Acknowledgment', 'Knowledge Deficit Question',
    'Common Ground Question', 'Metacomment', 'Solution Request']

print(f"{len(SCHEME)} acts loaded")

In [ ]:
# (2) Classifier signature.  <<< EDIT THE DOCSTRING BELOW to tune the task framing. >>>
class DialogueActClassifierSignature(dspy.Signature):
    """Given an utterance from a user in a tutoring or teaching conversation with an AI assistant (the user is learning a topic, or working through it with the assistant's help), first split it into clauses, then classify each clause into the dialogue acts that apply per the taxonomy provided. Output all applicable acts.

Disambiguating the question-type acts (these are easily confused — read carefully):
- "Solution Request": an OPEN request for the assistant to explain, teach, or give the answer, with NO specific candidate proposed. E.g. "explain how photosynthesis works", "teach me how to solve these", "what's the answer?", "walk me through it", "just tell me the steps".
- "Common Ground Question": the user proposes their OWN answer, understanding, or approach and asks the assistant to confirm/evaluate it. E.g. "so mitosis makes two identical cells, right?", "is my understanding correct?", "so it's because of gravity?", "does that mean the answer is 42?".
- "Knowledge Deficit Question": asks about a concept, term, definition, or a specific step — or clarifies something the assistant just said — NOT a request for the whole answer. E.g. "what does 'derivative' mean?", "why does that step work?", "what did you mean by that?".
- "Think Aloud": the user narrates their OWN reasoning or works through the problem out loud, rather than asking or reacting. E.g. "so if the cell loses water it should shrink...", "okay, 12 times 8 is 96, then I add 4".
- "Metacomment": the user comments on their own state of knowledge/understanding, not the content. E.g. "I don't get it", "oh, I see now", "I'm confused"."""
    utterance = dspy.InputField(desc="The utterance to be classified.")
    taxonomy = dspy.InputField(desc="Taxonomy of dialogue acts, with example utterances.")
    dialogue_acts: list[DialogueAct] = dspy.OutputField(desc="The list of dialogue acts applicable to this utterance.")

print(DialogueActClassifierSignature.__doc__.splitlines()[0])

In [ ]:
# (3) Panel suite + agreement metrics (copied from phase-2 dialogue_act_annotation.py)
class DialogueActSuite(dspy.Module):
    def __init__(self, arbiter=None, callbacks=None, lm_timeout=120, wall_timeout=150):
        super().__init__(callbacks)
        self.wall_timeout = wall_timeout
        specs = {
            "llama": dspy.LM("together_ai/meta-llama/Llama-3.3-70B-Instruct-Turbo", temperature=0.0, max_tokens=2048, timeout=lm_timeout),
            "gpt": dspy.LM("openai/gpt-5.4-mini", max_tokens=2048, timeout=lm_timeout),
            "sonnet": dspy.LM("anthropic/claude-sonnet-5", max_tokens=2048, timeout=lm_timeout),
        }
        self.annotators = {}
        for name, lm in specs.items():
            a = dspy.ChainOfThought(DialogueActClassifierSignature)
            a.set_lm(lm)
            self.annotators[name] = a
        self.arbiter = None
        if arbiter is not None:
            self.arbiter = dspy.ChainOfThought(DialogueActClassifierSignature)
            self.arbiter.set_lm(arbiter)

    def _annotate(self, name, annotator, utterance):
        try:
            acts = annotator(utterance=utterance, taxonomy=TAXONOMY).dialogue_acts
            valid = set(acts)
            return name, [k for k in SCHEME if k in valid]
        except Exception as e:
            _log.warning("dialogue-act annotator %r failed: %s: %s", name, type(e).__name__, e)
            return name, None

    def forward(self, utterance, min_votes=None):
        per_model = {}
        ex = concurrent.futures.ThreadPoolExecutor(max_workers=len(self.annotators))
        fut_to_name = {ex.submit(self._annotate, name, a, utterance): name
                       for name, a in self.annotators.items()}
        try:
            for fut in concurrent.futures.as_completed(fut_to_name, timeout=self.wall_timeout):
                name, acts = fut.result()
                per_model[name] = acts
        except concurrent.futures.TimeoutError:
            pass
        finally:
            for fut, name in fut_to_name.items():
                if name not in per_model:
                    fut.cancel()
                    _log.warning("dialogue-act annotator %r exceeded wall timeout %ss; dropping", name, self.wall_timeout)
                    per_model[name] = None
            ex.shutdown(wait=False)
        valid = [acts for acts in per_model.values() if acts is not None]
        n_valid = len(valid)
        votes = Counter(act for acts in valid for act in acts)
        if min_votes is None:
            min_votes = n_valid // 2 + 1 if n_valid else 1
        consensus = [k for k in SCHEME if votes[k] >= min_votes]
        confidence = {act: votes[act] / n_valid for act in votes} if n_valid else {}
        no_majority = not consensus and bool(votes)
        needs_review = no_majority or n_valid < 2
        final, arbiter_used = consensus, False
        if no_majority:
            if self.arbiter is not None:
                _, arb_acts = self._annotate("arbiter", self.arbiter, utterance)
                if arb_acts is not None:
                    final, arbiter_used = arb_acts, True
                else:
                    final = [k for k in SCHEME if votes[k] >= 1]
            else:
                final = [k for k in SCHEME if votes[k] >= 1]
        return {
            "final": final, "consensus": consensus, "votes": dict(votes),
            "confidence": confidence, "per_model": per_model, "n_valid": n_valid,
            "min_votes": min_votes, "no_majority": no_majority,
            "needs_review": needs_review, "arbiter_used": arbiter_used,
        }


def _complete_cases(records, raters):
    return [{r: set(rec[r]) for r in raters}
            for rec in records if all(rec.get(r) is not None for r in raters)]

def mean_pairwise_jaccard(records, raters=None):
    raters = raters or sorted({r for rec in records for r in rec})
    items = _complete_cases(records, raters)
    if not items or len(raters) < 2:
        return float("nan")
    pair_scores = []
    for it in items:
        for i in range(len(raters)):
            for j in range(i + 1, len(raters)):
                a, b = it[raters[i]], it[raters[j]]
                pair_scores.append(1.0 if not a and not b else len(a & b) / len(a | b))
    return sum(pair_scores) / len(pair_scores)

def fleiss_kappa(records, raters=None):
    raters = raters or sorted({r for rec in records for r in rec})
    items = _complete_cases(records, raters)
    n = len(raters)
    if len(items) < 2 or n < 2:
        return float("nan"), {}
    kappa_by_act = {}
    for act in SCHEME:
        counts = [sum(act in items[i][r] for r in raters) for i in range(len(items))]
        if all(c == 0 for c in counts) or all(c == n for c in counts):
            continue
        P = [(c * c + (n - c) ** 2 - n) / (n * (n - 1)) for c in counts]
        Pbar = sum(P) / len(P)
        p_present = sum(counts) / (len(counts) * n)
        Pe = p_present ** 2 + (1 - p_present) ** 2
        kappa_by_act[act] = 1.0 if Pe == 1 else (Pbar - Pe) / (1 - Pe)
    macro = sum(kappa_by_act.values()) / len(kappa_by_act) if kappa_by_act else float("nan")
    return macro, kappa_by_act

print("DialogueActSuite + metrics ready")

In [ ]:
# (4) Annotate assignment user-turns with OUR 5-act taxonomy.
import random
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

MAX_TURNS_PER_STUDENT = 20    # sample cap per student (cost control); set None for all
USE_PANEL = False
SEED = 0
random.seed(SEED)

keep_idx = []
for uid, g in turns_df.groupby("userId"):
    idx = list(g.index)
    keep_idx += (random.sample(idx, MAX_TURNS_PER_STUDENT)
                 if (MAX_TURNS_PER_STUDENT and len(idx) > MAX_TURNS_PER_STUDENT) else idx)
sub = turns_df.loc[keep_idx].reset_index(drop=True)
print(f"annotating {len(sub)} turns across {sub['userId'].nunique()} students"
      + ("  [PANEL x3]" if USE_PANEL else "  [single model]"))

if USE_PANEL:
    _suite = DialogueActSuite()
    def _classify(text): return _suite(utterance=text)["final"]
else:
    _single = dspy.ChainOfThought(DialogueActClassifierSignature)
    _single.set_lm(dspy.LM("openai/gpt-5.4-mini", temperature=0.0, max_tokens=2048, timeout=120))
    def _classify(text):
        acts = _single(utterance=text, taxonomy=TAXONOMY).dialogue_acts
        return [k for k in SCHEME if k in set(acts)]

def _run(i_text):
    i, text = i_text
    try: return i, _classify(text)
    except Exception: return i, None

with ThreadPoolExecutor(max_workers=8) as ex:
    res = list(tqdm(ex.map(_run, list(enumerate(sub["prompt"]))), total=len(sub), desc="acts"))
acts_list = [None] * len(sub)
for i, a in res: acts_list[i] = a
sub["acts"] = acts_list
sub.to_json("acts_studychat.json", orient="records", indent=2)
n_err = sub["acts"].isna().sum()
print(f"annotated {len(sub) - n_err}/{len(sub)} ({n_err} errors) -> acts_studychat.json")
sub.head()

In [ ]:
# (5) Outcome: does R1 reasoning predict exam (no_llm) transfer, controlling for assignment (llm)?
import numpy as np
import statsmodels.api as sm
from collections import Counter

REASON = {"Think Aloud", "Common Ground Question"}
ok = sub[sub["acts"].notna()].copy()
ok["reasoning"] = ok["acts"].map(lambda a: bool(set(a) & REASON))
ok["solreq"]    = ok["acts"].map(lambda a: "Solution Request" in a)

per = (ok.groupby("userId")
         .agg(reasoning_share=("reasoning", "mean"), solreq_share=("solreq", "mean"),
              n_turns=("reasoning", "size")).reset_index())
m = per.merge(grades[["userId", "llm", "no_llm"]], on="userId").dropna(subset=["no_llm", "llm"])

nt = len(ok)
ac = Counter(a for acts in ok["acts"] for a in acts)
print(f"{nt} turns annotated | {len(m)} students in outcome sample")
print("act coverage:", {k: f"{ac.get(k,0)/nt:.0%}" for k in SCHEME if ac.get(k, 0)})
print(f"corr(reasoning, assignment-llm)={np.corrcoef(m.reasoning_share, m.llm)[0,1]:+.2f}  "
      f"corr(assignment-llm, exam-no_llm)={np.corrcoef(m.llm, m.no_llm)[0,1]:+.2f}")

def ols(cols, names, y, lbl):
    z = lambda a: (a - np.mean(a)) / np.std(a, ddof=1)
    X = sm.add_constant(np.column_stack([z(np.asarray(c, float)) for c in cols]))
    r = sm.OLS(z(np.asarray(y, float)), X).fit(cov_type="HC3"); ci = r.conf_int()
    print(f"{lbl}  R2={r.rsquared:.3f}")
    for i, nm in enumerate(["const"] + names):
        if nm == "const": continue
        print(f"   {nm:<17} beta={r.params[i]:+.3f}  95%CI[{ci[i,0]:+.3f},{ci[i,1]:+.3f}]  p={r.pvalues[i]:.3f}")

print("\nDV = exam (no_llm), standardized, HC3 robust SE:")
ols([m.reasoning_share], ["reasoning"], m.no_llm, " reasoning only         ")
ols([m.reasoning_share, m.llm], ["reasoning", "assignment(llm)"], m.no_llm, " + assignment(llm) ctrl ")